# 실습02_핸드트래킹으로 그림그리기
-  MediaPipe Hands를 사용하여 손가락 움직임을 감지하고, 손가락 끝을 이용해 가상으로 그림을 그리는 프로그램
- OpenCV를 활용하여 손가락을 추적하고, 검출된 손가락 끝 위치를 기반으로 화면에 선을 그림


In [2]:
import cv2
import mediapipe as mp
import numpy as np

from pathlib import Path
import urllib.request
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
# HandLandmarker 모델 준비
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

HAND_LANDMARKER_MODEL = MODEL_DIR / 'hand_landmarker.task'
HAND_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'


def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return
    print(f'모델 다운로드 중: {model_path}')
    urllib.request.urlretrieve(model_url, model_path)


download_model(HAND_LANDMARKER_MODEL, HAND_LANDMARKER_MODEL_URL)

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
mp_hands = vision.HandLandmarksConnections
mp_drawing = vision.drawing_utils


def create_hand_landmarker(num_hands=1):
    # TODO: 손 검출/추적 옵션 설정
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(HAND_LANDMARKER_MODEL)),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=num_hands,
        min_hand_detection_confidence=0.5, # 손이 잘 안 잡히면 낮추고, 배경을 손으로 잘못 잡으면 높인다.
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5

    )
    return HandLandmarker.create_from_options(options)


def to_mp_image(frame):
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)


def landmark_to_pixel(landmark, frame_shape):
    # MediaPipe 좌표는 0~1 정규화 좌표이므로 실제 픽셀 좌표로 변환합니다.
    h, w = frame_shape[:2]
    x = int(landmark.x * w)
    y = int(landmark.y * h)
    return x, y

In [4]:
# 현재 그리기 상태
# 손이 사라졌다가 다시 나타나면 이전 위치와 갑자기 연결되지 않도록 False로 바꿈.
drawing = False

# 그림 누적 레이어
layer = None

In [5]:
# 직전 검지 끝 좌표
last_x, last_y = 0, 0

In [8]:
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_hand_landmarker(num_hands=1) as hand_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # TODO: 거울 보기용 좌우 반전
        frame = cv2.flip(frame, 1)

        # 첫 프레임 크기 레이어 생성
        if layer is None:
            layer = np.zeros_like(frame)

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            # TODO: 첫 번째 손의 검지 끝 사용
            hand_landmarks = result.hand_landmarks[0]
            index_tip = hand_landmarks[8]
            x, y = landmark_to_pixel(index_tip, frame.shape)

            # TODO: 검지 끝 표시
            cv2.circle(frame, (x, y), 8, (0, 255, 255), -1)

            if drawing:
                # TODO: 이전/현재 위치 연결
                cv2.line(layer, (last_x, last_y), (x, y), (255, 0, 255), 5)

            last_x, last_y = x, y
            drawing = True
        else:
            # 손 사라지면 그리기 중지
            drawing = False

        # TODO: 웹캠 영상과 그림 레이어 합성
        frame = cv2.add(frame, layer)
        cv2.imshow('Hand Tracking Drawing', frame)

        key = cv2.waitKey(5) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('c'):
            # c 입력 시 그림 지우기
            layer[:] = 0
            drawing = False

cap.release()
cv2.destroyAllWindows()

- 추가 기능 실습
    - 캔버스 지우기 기능 추가
    - 색상 변경